In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score

BACKEND = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Data" / "tourism_statistics.csv").is_file()), None)
if BACKEND is None: raise FileNotFoundError("Could not locate Backend/Data/tourism_statistics.csv")
sys.path.insert(0,str(BACKEND))
from ml_services.tourism import _tourism_rows, MONTHS
df=pd.read_csv(BACKEND/"Data"/"tourism_statistics.csv")
visits=_tourism_rows()
if not visits: raise ValueError("Monthly tourism workbook has no usable rows")
print("Destination profiles:",len(df),"| monthly tourism rows:",len(visits))

Destination profiles: 110 | monthly tourism rows: 432


In [2]:
base={"low":25,"medium":48,"moderate":48,"high":70,"very high":88}
def bucket(score):
    if score<36: return "Low"
    if score<61: return "Medium"
    return "High"
rows=[]
for r in df.itertuples(index=False):
    state_rows=[x for x in visits if x["state"].strip().casefold()==str(r.state).strip().casefold()]
    chosen=state_rows or visits
    monthly_mean={m:np.mean([x["total_tourists"] for x in chosen if x["month"]==m]) for m in MONTHS}
    monthly_mean={m:v for m,v in monthly_mean.items() if not np.isnan(v)}
    overall=float(np.mean(list(monthly_mean.values()))) if monthly_mean else 0
    crowd=str(r.crowd_level).strip().casefold()
    if crowd not in base or overall<=0: continue
    for month_num,month_name in enumerate(MONTHS,1):
        vals=[x["total_tourists"] for x in chosen if x["month"]==month_name]
        if not vals or month_name not in monthly_mean: continue
        month_index=float(np.mean(vals))/overall*100
        score=max(0,min(100,base[crowd]+(month_index-100)*.22))
        rows.append({"destination":r.destination,"crowd_reference":crowd,"category":str(r.category or "Unknown"),"region":str(r.region or "Unknown"),"budget":float(r.avg_budget_inr_per_person),"duration_days":float(r.avg_trip_duration_days),"month":month_num,"relative_month_index":month_index,"proxy_crowd":bucket(score)})
monthly=pd.DataFrame(rows)
print("rows:",len(monthly),"| profile groups:",monthly.destination.nunique())
print("class counts:",monthly.proxy_crowd.value_counts().reindex(["Low","Medium","High"],fill_value=0).to_dict())

rows: 1320 | profile groups: 110
class counts: {'Low': 309, 'Medium': 579, 'High': 432}


In [5]:
cat=["crowd_reference","category","region"]
num=["budget","duration_days","month","relative_month_index"]
features=cat+num
# StratifiedGroupKFold keeps entire destinations out of training while making
# held-out class proportions representative of the complete proxy dataset.
split=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)
train_i,test_i=next(split.split(monthly[features],monthly.proxy_crowd,groups=monthly.destination))
train,test=monthly.iloc[train_i],monthly.iloc[test_i]
prep=ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore"),cat),("num",StandardScaler(),num)])
model=make_pipeline(prep,LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42))
model.fit(train[features],train.proxy_crowd)
pred=model.predict(test[features]); probs=model.predict_proba(test[features]); labels=["Low","Medium","High"]
def scores(actual,guess,prob,classes):
    out={"accuracy":accuracy_score(actual,guess),"precision_macro":precision_score(actual,guess,labels=labels,average="macro",zero_division=0),"recall_macro":recall_score(actual,guess,labels=labels,average="macro",zero_division=0),"F1_macro":f1_score(actual,guess,labels=labels,average="macro",zero_division=0)}
    auc_labels=sorted(labels)
    order=[list(classes).index(x) for x in auc_labels]
    out["ROC-AUC_macro_OVR"]=roc_auc_score(actual,prob[:,order],labels=auc_labels,multi_class="ovr",average="macro") if set(labels).issubset(set(actual)) else None
    return out
print("Target: crowd category")
print("Train rows:",len(train),"| test rows:",len(test))
print("Train classes:",train.proxy_crowd.value_counts().to_dict(),"| test classes:",test.proxy_crowd.value_counts().to_dict())
print(pd.Series(scores(test.proxy_crowd,pred,probs,model[-1].classes_),name="Logistic Regression metrics").to_string())
print("\nConfusion matrix (rows=actual; columns=predicted; order Low, Medium, High):\n",confusion_matrix(test.proxy_crowd,pred,labels=labels))
print("\nClassification report:\n",classification_report(test.proxy_crowd,pred,labels=labels,zero_division=0))
import joblib
# After held-out evaluation, refit the same pipeline on all proxy rows for app inference.
service_model=make_pipeline(prep,LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42))
service_model.fit(monthly[features],monthly.proxy_crowd)
artifact_dir=BACKEND / "artifacts"
artifact_dir.mkdir(exist_ok=True)
joblib.dump(service_model,artifact_dir / "destination_crowd_logistic.joblib")
print("Saved trained model:",artifact_dir / "destination_crowd_logistic.joblib")


Target: crowd category
Train rows: 1056 | test rows: 264
Train classes: {'Medium': 463, 'High': 335, 'Low': 258} | test classes: {'Medium': 116, 'High': 97, 'Low': 51}
accuracy             0.962121
precision_macro      0.952475
recall_macro         0.969576
F1_macro             0.959917
ROC-AUC_macro_OVR    0.996418

Confusion matrix (rows=actual; columns=predicted; order Low, Medium, High):
 [[ 51   0   0]
 [  6 109   1]
 [  0   3  94]]

Classification report:
               precision    recall  f1-score   support

         Low       0.89      1.00      0.94        51
      Medium       0.97      0.94      0.96       116
        High       0.99      0.97      0.98        97

    accuracy                           0.96       264
   macro avg       0.95      0.97      0.96       264
weighted avg       0.96      0.96      0.96       264

Saved trained model: E:\Tourisense-New\Tourisense\Backend\artifacts\destination_crowd_logistic.joblib
